## EXP-ROUTER-001 — Router Agent & Model Selection

### Purpose

CEREBRO should not use an expensive frontier model for every AI task.

I want a Router Agent to decide whether a task can be handled by a local/cheaper model or requires a frontier model based on privacy, complexity, quality requirements, cost, and latency.

The routing decision must be explainable and traceable.

**Principle:** Use the lightest capable model. Escalate when necessary.

### 1. Define available model tiers

local    → actual available local model \
frontier → actual approved frontier model

In [ ]:
model_tiers = {
    "local": {
        "tier": "local",
        "privacy": "high",
        "relative_cost": "low",
        "relative_latency": "low",
        "capability": "standard"
    },

    "frontier": {
        "tier": "frontier",
        "privacy": "external_processing",
        "relative_cost": "high",
        "relative_latency": "variable",
        "capability": "advanced"
    }
}

model_tiers

### 2. Define routing signals

In [1]:
routing_signals = {
    "privacy_sensitive": False,
    "task_complexity": "low",
    "quality_requirement": "standard",
    "structured_output": True,
    "grounding_required": True,
    "cost_sensitive": True,
    "latency_sensitive": True
}
routing_signals

{'privacy_sensitive': False,
 'task_complexity': 'low',
 'quality_requirement': 'standard',
 'structured_output': True,
 'grounding_required': True,
 'cost_sensitive': True,
 'latency_sensitive': True}

### 3. Define the first routing policy

In [2]:
def route_task(signals):

    reasons = []

    # Privacy-sensitive tasks should remain local
    if signals["privacy_sensitive"]:
        reasons.append(
            "Artifact is privacy-sensitive; local processing preferred."
        )

        return {
            "route": "local",
            "reasons": reasons,
            "escalation_allowed": False
        }

    # Complex tasks may require frontier capability
    if signals["task_complexity"] == "high":
        reasons.append(
            "Task requires advanced reasoning capability."
        )

        return {
            "route": "frontier",
            "reasons": reasons,
            "escalation_allowed": False
        }

    # High-quality requirement may justify frontier processing
    if signals["quality_requirement"] == "high":
        reasons.append(
            "Task requires higher model capability."
        )

        return {
            "route": "frontier",
            "reasons": reasons,
            "escalation_allowed": False
        }

    # Otherwise prefer the cheaper/local model
    reasons.append(
        "Task is within standard local-model capability."
    )

    if signals["cost_sensitive"]:
        reasons.append(
            "Cost-sensitive processing favors local execution."
        )

    if signals["latency_sensitive"]:
        reasons.append(
            "Local execution may reduce external inference latency."
        )

    return {
        "route": "local",
        "reasons": reasons,
        "escalation_allowed": True
    }

### 4. Route our metadata enrichment task

In [3]:
routing_decision = route_task(
    routing_signals
)

routing_decision

{'route': 'local',
 'reasons': ['Task is within standard local-model capability.',
  'Cost-sensitive processing favors local execution.',
  'Local execution may reduce external inference latency.'],
 'escalation_allowed': True}

### 5. Human-readable explanation

In [4]:
print("CEREBRO Router Decision")
print("-----------------------")

print(
    "Selected tier:",
    routing_decision["route"].upper()
)

print("\nReasoning:")

for reason in routing_decision["reasons"]:
    print("-", reason)

print(
    "\nEscalation allowed:",
    routing_decision["escalation_allowed"]
)

CEREBRO Router Decision
-----------------------
Selected tier: LOCAL

Reasoning:
- Task is within standard local-model capability.
- Cost-sensitive processing favors local execution.
- Local execution may reduce external inference latency.

Escalation allowed: True


### 6. Create routing provenance

In [6]:
routing_record = {
    "router": "cerebro_router",
    "router_version": "0.1",

    "task_id": enrichment_task["task_id"]
        if "enrichment_task" in globals()
        else "TASK-ENRICH-001",

    "signals": routing_signals,

    "decision": {
        "route": routing_decision["route"],
        "reasons": routing_decision["reasons"]
    },

    "execution": {
        "model": None,
        "status": "not_executed"
    },

    "escalation": {
        "allowed": routing_decision[
            "escalation_allowed"
        ],
        "triggered": False,
        "reason": None
    }
}

routing_record

{'router': 'cerebro_router',
 'router_version': '0.1',
 'task_id': 'TASK-ENRICH-001',
 'signals': {'privacy_sensitive': False,
  'task_complexity': 'low',
  'quality_requirement': 'standard',
  'structured_output': True,
  'grounding_required': True,
  'cost_sensitive': True,
  'latency_sensitive': True},
 'decision': {'route': 'local',
  'reasons': ['Task is within standard local-model capability.',
   'Cost-sensitive processing favors local execution.',
   'Local execution may reduce external inference latency.']},
 'execution': {'model': None, 'status': 'not_executed'},
 'escalation': {'allowed': True, 'triggered': False, 'reason': None}}

### Cell 7. Define validation gates

In [7]:
validation_gates = {
    "structured_output_valid": True,
    "grounding_passed": True,
    "required_fields_present": True,
    "unsupported_claims_detected": False,
    "entity_preservation_passed": True
}

validation_gates

{'structured_output_valid': True,
 'grounding_passed': True,
 'required_fields_present': True,
 'unsupported_claims_detected': False,
 'entity_preservation_passed': True}

### 8. Determine whether escalation is required

In [8]:
def should_escalate(
    routing_record,
    validation_gates
):

    if not routing_record["escalation"]["allowed"]:
        return False, None

    failures = []

    if not validation_gates["structured_output_valid"]:
        failures.append("invalid_structured_output")

    if not validation_gates["grounding_passed"]:
        failures.append("grounding_failure")

    if not validation_gates["required_fields_present"]:
        failures.append("missing_required_fields")

    if validation_gates["unsupported_claims_detected"]:
        failures.append("unsupported_claims")

    if not validation_gates["entity_preservation_passed"]:
        failures.append("entity_preservation_failure")

    if failures:
        return True, failures

    return False, None

In [9]:
escalate, escalation_reason = should_escalate(
    routing_record,
    validation_gates
)

print("Escalate:", escalate)
print("Reason  :", escalation_reason)

Escalate: False
Reason  : None


### 9. Test failure/escalation behavior

In [10]:
failed_validation = validation_gates.copy()

failed_validation["grounding_passed"] = False

escalate, escalation_reason = should_escalate(
    routing_record,
    failed_validation
)

print("Escalate:", escalate)
print("Reason  :", escalation_reason)

Escalate: True
Reason  : ['grounding_failure']


### 10. Simulate escalation record

In [11]:
if escalate:

    routing_record["escalation"][
        "triggered"
    ] = True

    routing_record["escalation"][
        "reason"
    ] = escalation_reason

    routing_record["decision"][
        "final_route"
    ] = "frontier"

else:

    routing_record["decision"][
        "final_route"
    ] = routing_record["decision"]["route"]

routing_record

{'router': 'cerebro_router',
 'router_version': '0.1',
 'task_id': 'TASK-ENRICH-001',
 'signals': {'privacy_sensitive': False,
  'task_complexity': 'low',
  'quality_requirement': 'standard',
  'structured_output': True,
  'grounding_required': True,
  'cost_sensitive': True,
  'latency_sensitive': True},
 'decision': {'route': 'local',
  'reasons': ['Task is within standard local-model capability.',
   'Cost-sensitive processing favors local execution.',
   'Local execution may reduce external inference latency.'],
  'final_route': 'frontier'},
 'execution': {'model': None, 'status': 'not_executed'},
 'escalation': {'allowed': True,
  'triggered': True,
  'reason': ['grounding_failure']}}

### 11. Reset and test normal path

In [12]:
routing_record["escalation"]["triggered"] = False
routing_record["escalation"]["reason"] = None

escalate, escalation_reason = should_escalate(
    routing_record,
    validation_gates
)

if escalate:

    final_route = "frontier"

else:

    final_route = routing_record[
        "decision"
    ]["route"]

routing_record["decision"][
    "final_route"
] = final_route

print("Initial route :", routing_record["decision"]["route"])
print("Escalated     :", escalate)
print("Final route   :", final_route)

Initial route : local
Escalated     : False
Final route   : local


### 12. Final acceptance test

In [13]:
assert routing_record["router"] == "cerebro_router"

assert (
    routing_record["decision"]["route"]
    == "local"
)

assert (
    routing_record["decision"]["final_route"]
    == "local"
)

assert (
    routing_record["escalation"]["allowed"]
    is True
)

assert (
    routing_record["escalation"]["triggered"]
    is False
)

print("✓ Router decision generated")
print("✓ Routing reasons recorded")
print("✓ Local-first policy applied")
print("✓ Validation gates supported")
print("✓ Frontier escalation tested")
print("✓ Final route traceable")

print("\nEXP-ROUTER-001: PASS")

✓ Router decision generated
✓ Routing reasons recorded
✓ Local-first policy applied
✓ Validation gates supported
✓ Frontier escalation tested
✓ Final route traceable

EXP-ROUTER-001: PASS


### Result

CEREBRO can now make an explainable model-tier routing decision before AI execution.

For this metadata-enrichment task, the Router selected local processing because the task is low complexity and cost-sensitive. The experiment also demonstrated that a failed validation gate can trigger escalation to a frontier model.

No specific model has been hard-wired into the pipeline.

The next step is to connect the Router to a model registry and execute the selected model so that routing can be evaluated using actual quality, latency, and cost measurements.

flowchart LR
    T[Task] --> R[Router Agent]
    R --> L[Local / Cheap]
    L --> V{Validation}
    V -->|Pass| O[Output]
    V -->|Fail| F[Frontier]
    F --> V2[Validate]
    V2 --> O